# Lensless Computational Imaging — Demo

This notebook demonstrates how to use our lensless image reconstruction models.

**What it does:**
1. Clones the repository and installs dependencies.
2. Downloads the trained model checkpoint from HuggingFace.
3. Downloads a user-provided `.zip` dataset from Google Drive and unzips it.
4. Runs `inference.py` to reconstruct images and saves them to disk.
5. Visualizes samples: ground truth (if available) vs lensless measurement vs reconstruction.
6. Runs `calculate_metrics.py` to report PSNR, SSIM, MSE, and LPIPS (only if ground truth exists).

**Expected dataset structure inside the zip:**
```
data/
  lensless/   ImageID1.png ...
  masks/      ImageID1.npy ...
  lensed/     ImageID1.png ...   # optional ground truth
```
Only `lensless/` and `masks/` are required. The PSF is simulated from each mask on the fly.

## 1. Clone repository and install dependencies

In [ ]:
!git clone https://github.com/AlexanderPlotnikovv/LenslessComputationalImaging.git
%cd LenslessComputationalImaging
!pip install -q -r requirements.txt

## 2. Download the trained checkpoint from HuggingFace

We use our best model (`le_admm_pre_post`) for the demo. The checkpoint is stored on the HuggingFace Hub.

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "AlexPlotnikovTech/lensless-computational-imaging"

ckpt_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="le_admm_pre_post/model_best.pth",
    local_dir="models",
)
print(f"Checkpoint downloaded to: {ckpt_path}")

## 3. Download and unzip the user-provided dataset

Paste a Google Drive URL to your `.zip` dataset below. The notebook downloads it,
unzips it, and automatically locates the folder that contains `lensless/`.

In [ ]:
import gdown
import zipfile
from pathlib import Path

# === USER INPUT: paste your Google Drive zip URL here ===
DATASET_URL = "https://drive.google.com/file/d/12H176havpGoF5gBQ0L7IyrngN-ctLKAp/view?usp=sharing"
# ========================================================

zip_path = "user_data.zip"
gdown.download(DATASET_URL, zip_path, quiet=False, fuzzy=True)

extract_dir = Path("user_data")
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

# locate the directory that actually contains lensless/
data_dir = None
for p in extract_dir.rglob("lensless"):
    data_dir = p.parent
    break
assert data_dir is not None, "Could not find a lensless/ folder inside the zip"
print(f"Dataset extracted to: {data_dir}")

## 4. Run inference

This reconstructs every image in the dataset and saves the results to
`data/saved/demo/test/<ImageID>.png`. Each reconstruction keeps the same ID as its
input image so they can be matched for evaluation.

In [ ]:
SAVE_PATH = "demo"

!python3 inference.py \
model = le_admm_pre_post \
    datasets = custom_dir \
               + inferencer.data_dir = {data_dir} \
    inferencer.from_pretrained = models / le_admm_pre_post / model_best.pth \
    inferencer.save_path = {SAVE_PATH} \
    inferencer.device = auto \
    dataloader.batch_size = 4 \
    dataloader.num_workers = 2

from pathlib import Path

RECON_DIR = Path("data/saved") / SAVE_PATH / "test"
print(f"Reconstructions saved to: {RECON_DIR}")

## 5. Visualize samples

For a few samples we show, side by side: the ground truth (if present), the raw
lensless measurement, and our reconstruction.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

recon_paths = sorted(RECON_DIR.glob("*.png"))[:4]  # show first 4 samples
has_gt = (data_dir / "lensed").exists()
n_cols = 3 if has_gt else 2

fig, axes = plt.subplots(len(recon_paths), n_cols,
                         figsize=(4 * n_cols, 4 * len(recon_paths)))
if len(recon_paths) == 1:
    axes = axes[None, :]

for row, recon_path in enumerate(recon_paths):
    image_id = recon_path.stem
    lensless = Image.open(data_dir / "lensless" / f"{image_id}.png")
    recon = Image.open(recon_path)

    col = 0
    if has_gt:
        gt = Image.open(data_dir / "lensed" / f"{image_id}.png")
        axes[row, col].imshow(gt)
        axes[row, col].set_title("Ground truth")
        axes[row, col].axis("off")
        col += 1

    axes[row, col].imshow(lensless)
    axes[row, col].set_title("Lensless measurement")
    axes[row, col].axis("off")
    col += 1

    axes[row, col].imshow(recon)
    axes[row, col].set_title("Reconstruction")
    axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 6. Compute metrics

If the dataset includes ground truth images (`lensed/`), we compute PSNR, SSIM, MSE,
and LPIPS over all reconstructed samples. Otherwise this step is skipped.

In [ ]:
for pred_path in tqdm(pred_paths):
    gt_path = gt_dir / pred_path.name
    if not gt_path.exists():
        continue

    pred = load_image(pred_path).unsqueeze(0)
    gt = load_image(gt_path, size=pred.shape[-2:]).unsqueeze(0)

    psnr_vals.append(peak_signal_noise_ratio(pred, gt, data_range=1.0).item())
    ssim_vals.append(structural_similarity_index_measure(pred, gt, data_range=1.0).item())
    mse_vals.append(torch.nn.functional.mse_loss(pred, gt).item())
    lpips_vals.append(lpips_fn(pred * 2 - 1, gt * 2 - 1).mean().item())